# Step 8: Evaluation + Reasons
Read the top flags; plant **seen** and **unseen** anomalies and report detection rate separately (MF3); compare with the rules baseline at the same number of flags (MF4); flagged real rows are "unverified"; compute reasons and finalize EN/AR texts.

**Input:** `data/processed/features.csv + models/isolation_forest.joblib`  
**Output:** `evaluation results, final reason texts`

In [1]:
# 8.1 Load the saved model and its settings from Step 7, and the reason texts (EN + AR) from src/predict.py,
# the same texts the ML service returns, so there's one copy of them.
# model_input() and apply_rules() are copied from notebook 06 for now; in Part B they move to src/features.py.
import json
import sys
import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, "..")            # lets the notebook import from the ml/src folder
from src.predict import REASONS     # code -> (English text, Arabic text)

df = pd.read_csv("../data/processed/features.csv", parse_dates=["expense_date"])
model = joblib.load("../models/isolation_forest.joblib")
with open("../models/model_config.json", encoding="utf-8") as fh:
    cfg = json.load(fh)
MODEL_FEATURES = cfg["model_features"]
print(cfg["model_version"], "|", len(df), "requests |", len(REASONS), "reason codes")

def model_input(data):
    # One-sided amounts (Step 7, 7.8), using the floors saved in the config
    X = data[MODEL_FEATURES].copy()
    X["log_amount"] = X["log_amount"].clip(lower=cfg["model_input"]["log_amount_floor"])
    X["amount_vs_employee_category"] = X["amount_vs_employee_category"].clip(lower=0)
    return X

def apply_rules(data):
    # The 3 rules (Step 7, 7.13), using the settings and limits saved in the config
    r, limits = cfg["rules"], cfg["category_limits"]
    same_day = data.groupby(["requester_id", "expense_date", "category"])["amount_sar"]
    total, smallest, largest = same_day.transform("sum"), same_day.transform("min"), same_day.transform("max")
    limit = data["category"].map(limits)
    n = data["same_category_day_count"]
    split = ((n >= 2) & (n <= r["split_max_parts"]) & (largest <= limit) & (total > limit)
             & (smallest >= r["split_min_part"] * limit))
    duplicate = (data["duplicate_count"].between(2, r["duplicate_max_copies"])
                 & (data["amount_sar"] >= r["duplicate_min_amount"]))
    many = data["other_category_day_count"] >= r["many_other_from"]
    return pd.DataFrame({"possible_split_request": split, "duplicate_request": duplicate,
                         "many_requests_same_day": many}, index=data.index)

iforest-0.2 | 209012 requests | 13 reason codes


_Code for this step comes next._

In [2]:
# 8.2 Same scoring as notebook 06 (7.15), from the saved files only.
X = model_input(df)
df["score"] = -model.score_samples(X)
rules = apply_rules(df)
cut = cfg["cut_offs"]
df["risk_level"] = np.where(df["score"] >= cut["high_from"], "high",
                   np.where((df["score"] >= cut["medium_from"]) | rules.any(axis=1), "medium", "low"))
print(df["risk_level"].value_counts())

risk_level
low       201444
medium      5476
high        2092
Name: count, dtype: int64


In [3]:
# 8.3 For every flagged request and every model feature: set that feature to a NORMAL value,
# score again, and measure how much the score dropped. A big drop = that feature is a reason.
# NORMAL values = what a typical request has: the amount floor, 0 spreads above normal,
# the median category share, not weekend, not round.
flagged = df["risk_level"] != "low"
X_flag = X[flagged]
score_flag = df.loc[flagged, "score"]

NORMAL = {
    "log_amount": cfg["model_input"]["log_amount_floor"],
    "amount_vs_employee_category": 0.0,
    "category_share_for_employee": float(X["category_share_for_employee"].median()),
    "is_weekend": 0,
    "is_round_amount": 0,
}
drop = pd.DataFrame(index=X_flag.index)
for feature, normal_value in NORMAL.items():
    what_if = X_flag.copy()
    what_if[feature] = normal_value                       # only this one feature changes
    drop[feature] = score_flag - (-model.score_samples(what_if))
print(drop.describe().T[["mean", "50%", "max"]].round(3))

                              mean    50%    max
log_amount                   0.020  0.008  0.224
amount_vs_employee_category  0.011  0.000  0.291
category_share_for_employee  0.015  0.012  0.087
is_weekend                   0.030  0.000  0.173
is_round_amount              0.019  0.000  0.203


In [4]:
# 8.4 Reasons for each flagged request:
#  1. Every rule that fired is a reason (strength 1.0, so rules come first: they're plain facts).
#  2. A model feature is a reason when its "what if" drop is at least MIN_DROP AND it makes sense:
#     the amount is really above normal, the category is really rare for THIS person (they have
#     their own history), the day really is a weekend, the amount really is round.
#     The amount reason's code depends on what it was compared with (baseline_level, Step 6).
#  3. Strongest first, at most 3 (the contract's limit). If nothing passes MIN_DROP, the single
#     strongest sensible feature is used, so every flagged request has at least one reason.
MIN_DROP = 0.02
MAX_REASONS = 3
UNUSUAL_SHARE = 0.2          # "rare category" = under 20% of this person's earlier requests
AMOUNT_CODE = {"empcat": "amount_high_for_employee_category",
               "emp": "amount_high_for_employee",
               "deptcat": "amount_high_for_department"}

def reasons_for(row, drops, fired_rules):
    candidates = {code: 1.0 for code, fired in fired_rules.items() if fired}
    checks = {   # reason code: (feature it comes from, does this reason make sense here?)
        "large_amount": ("log_amount", row["log_amount"] > cfg["model_input"]["log_amount_floor"]),
        AMOUNT_CODE.get(row["baseline_level"], "none"): ("amount_vs_employee_category",
            row["amount_vs_employee_category"] >= 1 and row["baseline_level"] in AMOUNT_CODE),
        "unusual_category_for_employee": ("category_share_for_employee",
            row["category_share_for_employee"] <= UNUSUAL_SHARE and row["baseline_level"] in ("empcat", "emp")),
        "weekend_expense": ("is_weekend", row["is_weekend"] == 1),
        "round_amount": ("is_round_amount", row["is_round_amount"] == 1),
    }
    model_candidates = {code: drops[feature] for code, (feature, makes_sense) in checks.items() if makes_sense}
    candidates.update({code: d for code, d in model_candidates.items() if d >= MIN_DROP})
    if not candidates and model_candidates:          # fallback: the strongest sensible one
        best = max(model_candidates, key=model_candidates.get)
        candidates[best] = model_candidates[best]
    return sorted(candidates, key=candidates.get, reverse=True)[:MAX_REASONS]

flag_rows = df[flagged]
codes = pd.Series([reasons_for(row, drop.loc[i], rules.loc[i]) for i, row in flag_rows.iterrows()],
                  index=flag_rows.index)
df["reason_codes"] = pd.Series([[] for _ in range(len(df))], index=df.index)   # low risk = no reasons
df.loc[flagged, "reason_codes"] = codes

print("Flagged:", flagged.sum(), "| without a reason:", (codes.str.len() == 0).sum())
print("Reasons per flagged request:", codes.str.len().value_counts().sort_index().to_dict())
print(codes.explode().value_counts())

Flagged: 7568 | without a reason: 0
Reasons per flagged request: {1: 2812, 2: 3699, 3: 1057}
large_amount                         2761
duplicate_request                    2569
weekend_expense                      1924
possible_split_request               1415
amount_high_for_employee_category    1218
unusual_category_for_employee        1134
round_amount                          980
many_requests_same_day                802
amount_high_for_employee              458
amount_high_for_department            120
Name: count, dtype: int64


In [5]:
# 8.5 The exact shape the ML service returns and the backend saves in risk_justification:
# [{"code", "text_en", "text_ar"}, ...]
def reason_details(code_list):
    return [{"code": c, "text_en": REASONS[c][0], "text_ar": REASONS[c][1]} for c in code_list]

df["reason_details"] = df["reason_codes"].apply(reason_details)

ex = df.sort_values("score", ascending=False).iloc[2]          # the 3rd most unusual request
print(ex["requester_id"], ex["category"], ex["amount_sar"], ex["risk_level"])
for d in ex["reason_details"]:
    print(" -", d["code"], "|", d["text_en"], "|", d["text_ar"])

3525 Travel 1125.0 high
 - amount_high_for_employee_category | Amount is much higher than this employee usually requests in this category | المبلغ أعلى بكثير مما يطلبه هذا الموظف عادةً في هذه الفئة
 - round_amount | Round amount (e.g. 500 or 1,000) | المبلغ رقم مقرّب (مثل 500 أو 1,000)
 - weekend_expense | Expense date falls on a weekend (Friday–Saturday) | تاريخ المصروف في عطلة نهاية الأسبوع (الجمعة–السبت)


In [6]:
# 8.6 The 6 most unusual requests and 5 random medium ones, with their reasons.
cols = ["requester_id", "category", "amount_sar", "risk_level", "reason_codes"]
print(df.sort_values("score", ascending=False)[cols].head(6).to_string(index=False))
print(df[df["risk_level"] == "medium"].sample(5, random_state=1)[cols].to_string(index=False))

 requester_id                category  amount_sar risk_level                                                       reason_codes
         3440 Equipment & Maintenance    183750.0       high    [round_amount, large_amount, amount_high_for_employee_category]
         1197            IT & Telecom     18750.0       high             [round_amount, large_amount, amount_high_for_employee]
         3525                  Travel      1125.0       high [amount_high_for_employee_category, round_amount, weekend_expense]
         4020                  Travel      5625.0       high [round_amount, amount_high_for_employee_category, weekend_expense]
         2878            IT & Telecom      9375.0       high          [round_amount, amount_high_for_employee, weekend_expense]
         5235            IT & Telecom     25125.0       high                      [round_amount, weekend_expense, large_amount]
 requester_id                category  amount_sar risk_level                                         rea

## Part B: One copy of the feature code
The feature code from notebook 04 (and the model input + rules from 06) now lives in `src/features.py`.
Every notebook from here on, and the ML service in Step 9, imports it, so planted test requests and live
requests get features computed exactly like the training data. Check: rebuilding features.csv gives identical numbers.

In [7]:
# 8.7 Import the shared feature code and rebuild every feature from requests.csv.
# If features.py is a faithful copy of notebook 04, the result must be IDENTICAL to features.csv:
# same rows in the same order, same numbers, same baseline_level.
import time
from src import features as F       # the ml/src folder is already on the path (8.1)

start = time.time()
requests = pd.read_csv("../data/processed/requests.csv", parse_dates=["expense_date", "created_at"])
rebuilt = F.build_features(requests)
print("Built:", rebuilt.shape, f"in {time.time() - start:.1f} s")

saved = pd.read_csv("../data/processed/features.csv", parse_dates=["expense_date"])
print("Same rows, same order:", rebuilt["expense_id"].equals(saved["expense_id"]))
print("Largest difference per feature:")
print((rebuilt[F.FEATURES] - saved[F.FEATURES]).abs().max())
print("baseline_level identical:", (rebuilt["baseline_level"] == saved["baseline_level"]).all())

Built: (209012, 21) in 1.7 s
Same rows, same order: True
Largest difference per feature:
log_amount                     0.0
amount_vs_employee             0.0
amount_vs_employee_category    0.0
amount_vs_department           0.0
category_share_for_employee    0.0
same_category_day_count        0.0
duplicate_count                0.0
daily_request_count            0.0
other_category_day_count       0.0
is_weekend                     0.0
is_round_amount                0.0
dtype: float64
baseline_level identical: True


In [8]:
# 8.8 Same scoring as 8.2, but with model_input / apply_rules / risk_levels from features.py
# instead of the copies in 8.1. Same counts = the shared versions are correct too.
scores = -model.score_samples(F.model_input(rebuilt, cfg))
levels = F.risk_levels(scores, F.apply_rules(rebuilt, cfg), cfg)
print(pd.Series(levels).value_counts())

low       199658
medium      7262
high        2092
Name: count, dtype: int64


## Part C: Planted "seen" anomalies (MF3, MF4)
No real request is labelled as fraud, so we plant suspicious requests ourselves (patterns the features were designed for),
build their features exactly like real ones, and count how many each system catches: model only, rules only,
simple policy rules without ML, and the hybrid. A planted case counts as caught if at least one of its requests is flagged.

In [9]:
# 8.9 Planted requests go to "normal" employees: at least 30 requests of history and a usual day of
# 1 request (so a planted split or busy day really stands out for THEM).
# For each one: department, main category, the average and spread of their log amounts, and the
# categories they've used. Bulk buyers are left out, like in training.
LIMITS = cfg["category_limits"]
CATEGORIES = list(LIMITS)

busy = F.find_bulk_buyers(requests)
history = requests[~requests["requester_id"].isin(busy)].copy()
history["log_amount"] = np.log1p(history["amount_sar"])
per_day = history.groupby(["requester_id", "expense_date"]).size()

profile = history.groupby("requester_id").agg(
    requests=("expense_id", "size"),
    department_id=("department_id", "last"),
    main_category=("category", lambda s: s.value_counts().index[0]),
    log_mean=("log_amount", "mean"),
    log_std=("log_amount", "std"),
    categories=("category", lambda s: set(s)),
)
profile["usual_day"] = per_day.groupby("requester_id").median()
eligible = profile[(profile["requests"] >= 30) & (profile["usual_day"] == 1)]
print("Employees who can receive planted requests:", len(eligible))
print("Typical spread of an employee's log amounts:", round(eligible["log_std"].median(), 2))

Employees who can receive planted requests: 1620
Typical spread of an employee's log amounts: 1.29


In [10]:
# 8.10 Six patterns, each planted for different employees on a date in Apr–Jun 2026 when they had
# no requests (so planted cases never mix with real ones or with each other):
#   too_big          1 request, k spreads above the person's normal (k = 2, 3, 4, 5, 6; 50 cases each)
#   split            3 requests in their main category, each 70–95% of the limit (together over it)
#   duplicate        the same amount (>= 1,000) twice on one day
#   new_category_big first request ever in a category, 3 spreads above their normal
#   weekend_round    a round SAR amount on Friday/Saturday, 2 spreads above normal
#   many_requests    10 normal-sized requests on one day across all 6 categories
# amount_original = the amount as typed: SAR for planted requests (USD for the PCard history),
# so is_round_amount is checked on the right currency (Step 5, 5.12).
rng = np.random.default_rng(42)
PLAN = ([("too_big", k) for k in (2, 3, 4, 5, 6) for _ in range(50)]
        + [(p, None) for p in ("split", "duplicate", "new_category_big", "weekend_round", "many_requests")
           for _ in range(100)])
employees = rng.choice(eligible.index.to_numpy(), size=len(PLAN), replace=False)

window = pd.date_range("2026-04-01", "2026-06-27")
weekdays = window[~window.day_of_week.isin(F.WEEKEND_DAYS)]
weekend_days = window[window.day_of_week.isin(F.WEEKEND_DAYS)]
taken = set(zip(history["requester_id"], history["expense_date"]))     # (employee, day) with real requests

rows = []
for case_id, ((pattern, k), emp) in enumerate(zip(PLAN, employees), start=1):
    p = eligible.loc[emp]
    cat = p["main_category"]
    days = weekend_days if pattern == "weekend_round" else weekdays
    day = pd.Timestamp(rng.choice(days))
    while (emp, day) in taken:                                          # pick a free day
        day = pd.Timestamp(rng.choice(days))

    def spreads_above(s):
        # the SAR amount that is s spreads above this person's normal (back from the log scale)
        return float(np.expm1(p["log_mean"] + s * p["log_std"]))

    def add(category, amount):
        rows.append({"requester_id": emp, "department_id": p["department_id"], "category": category,
                     "amount_sar": round(amount, 2), "expense_date": day,
                     "pattern": pattern, "spreads": k, "case_id": case_id})

    if pattern == "too_big":
        add(cat, spreads_above(k) + 0.37)                  # + cents so it isn't accidentally round
    elif pattern == "split":
        for _ in range(3):
            add(cat, LIMITS[cat] * rng.uniform(0.70, 0.95))
    elif pattern == "duplicate":
        amount = max(1000.0, spreads_above(1)) + 0.45
        add(cat, amount)
        add(cat, amount)
    elif pattern == "new_category_big":
        never = [c for c in CATEGORIES if c not in p["categories"]] or [c for c in CATEGORIES if c != cat]
        add(rng.choice(never), spreads_above(3) + 0.21)
    elif pattern == "weekend_round":
        add(cat, max(500.0, round(spreads_above(2), -2)))  # round(…, -2) = nearest 100
    elif pattern == "many_requests":
        for c, times in zip(CATEGORIES, (2, 2, 2, 2, 1, 1)):
            for _ in range(times):
                add(c, spreads_above(0) * rng.uniform(0.7, 1.3))

planted = pd.DataFrame(rows)
planted["expense_id"] = np.arange(900001, 900001 + len(planted))      # ids no real request uses
planted["amount_original"] = planted["amount_sar"]
print(planted.groupby("pattern").agg(cases=("case_id", "nunique"), requests=("expense_id", "size")))

                  cases  requests
pattern                          
duplicate           100       200
many_requests       100      1000
new_category_big    100       100
split               100       300
too_big             250       250
weekend_round       100       100


In [11]:
# 8.11 Put the planted requests among the real ones and build ALL features with features.py,
# exactly as for real data: earlier days only, the new-employee rule, same-day counts.
# Real rows keep amount_original = USD; planted rows use SAR.
real = requests.assign(amount_original=requests["amount_usd"], pattern="real", spreads=np.nan, case_id=0)
combined = pd.concat([real, planted], ignore_index=True)
feats = F.build_features(combined, round_column="amount_original")
feats = feats.merge(combined[["expense_id", "pattern", "spreads", "case_id"]], on="expense_id", how="left")
is_real = feats["pattern"] == "real"
print("Real requests:", is_real.sum(), "| planted:", (~is_real).sum())

# Did "k spreads above normal" really come out as k? (median of the feature per k)
print(feats[feats["pattern"] == "too_big"].groupby("spreads")["amount_vs_employee_category"].median().round(2).to_dict())

Real requests: 209012 | planted: 1950
{2.0: 2.06, 3.0: 3.19, 4.0: 4.33, 5.0: 5.17, 6.0: 6.35}


In [12]:
# 8.12 Each system's yes/no per request. To compare fairly (MF4), the model alone gets the SAME
# number of flags on real data as the hybrid: its cut-off = the score that leaves that share above it.
scores = -model.score_samples(F.model_input(feats, cfg))
rules = F.apply_rules(feats, cfg)
feats["hybrid"] = F.risk_levels(scores, rules, cfg) != "low"
feats["rules_only"] = rules.any(axis=1)

budget = feats.loc[is_real, "hybrid"].mean()                          # share of real requests flagged
feats["model_only"] = scores >= np.quantile(scores[is_real], 1 - budget)

limit = feats["category"].map(LIMITS)
feats["policy_rules"] = (feats["rules_only"]
                         | (feats["amount_sar"] > limit)                               # over the limit
                         | (feats["is_weekend"] == 1)                                  # any weekend
                         | ((feats["is_round_amount"] == 1) & (feats["amount_sar"] >= 1000)))   # big round

SYSTEMS = ["model_only", "rules_only", "policy_rules", "hybrid"]
print("Share of REAL requests flagged (%):")
print((feats.loc[is_real, SYSTEMS].mean() * 100).round(2))

Share of REAL requests flagged (%):
model_only       4.47
rules_only       3.08
policy_rules    16.43
hybrid           4.47
dtype: float64


In [13]:
# 8.13 A planted case is caught if at least ONE of its requests is flagged (the manager sees it).
cases = (feats[~is_real]
         .groupby(["pattern", "spreads", "case_id"], dropna=False)[SYSTEMS].any()
         .reset_index())
table = cases.groupby("pattern")[SYSTEMS].mean().mul(100).round(0)
table = table.reindex(["too_big", "split", "duplicate", "new_category_big", "weekend_round", "many_requests"])
table.loc["ALL"] = cases[SYSTEMS].mean().mul(100).round(0)
print(table)

                  model_only  rules_only  policy_rules  hybrid
pattern                                                       
too_big                 78.0         5.0          90.0    69.0
split                    9.0       100.0         100.0   100.0
duplicate                2.0       100.0         100.0   100.0
new_category_big       100.0         3.0          79.0    70.0
weekend_round          100.0         2.0         100.0   100.0
many_requests            2.0       100.0         100.0   100.0
ALL                     55.0        42.0          94.0    86.0


In [14]:
# 8.14 Detection of "too big" by how many spreads above the person's normal it was.
curve = cases[cases["pattern"] == "too_big"].groupby("spreads")[SYSTEMS].mean().mul(100).round(0)
print(curve)

         model_only  rules_only  policy_rules  hybrid
spreads                                              
2.0            10.0         2.0          60.0     4.0
3.0            82.0         0.0          94.0    46.0
4.0           100.0        10.0          96.0    94.0
5.0           100.0         8.0          98.0   100.0
6.0           100.0         4.0         100.0   100.0


## Part D: Unseen anomalies (MF3)
Suspicious requests invented by an independent AI agent that had no access to the model's design, only a one-page task
and the request history. Same method as Part C. The results are saved BEFORE the answers file is opened.

In [15]:
# 8.15 The unseen requests: only case_id, employee, department, category, SAR amount, date. No hints.
# New ids that no real or Part C request uses; amounts were typed in SAR, so round is checked on SAR.
unseen = pd.read_csv("../data/unseen/unseen_requests.csv", parse_dates=["expense_date"])
unseen["expense_id"] = np.arange(950001, 950001 + len(unseen))
unseen["amount_original"] = unseen["amount_sar"]
print("Unseen:", len(unseen), "requests in", unseen["case_id"].nunique(), "cases")
print("Requests per case:", unseen.groupby("case_id").size().value_counts().sort_index().to_dict())

Unseen: 284 requests in 151 cases
Requests per case: {1: 58, 2: 53, 3: 40}


In [16]:
# 8.16 Exactly the Part C method: put the unseen requests among the real ones, build features with
# features.py, then the 4 systems. The model alone again gets the hybrid's share of flags on real data.
real = requests.assign(amount_original=requests["amount_usd"], case_id=0)
combined = pd.concat([real, unseen], ignore_index=True)
feats = F.build_features(combined, round_column="amount_original")
feats = feats.merge(combined[["expense_id", "case_id"]], on="expense_id", how="left")
is_real = feats["case_id"] == 0

scores = -model.score_samples(F.model_input(feats, cfg))
rules_all = F.apply_rules(feats, cfg)
feats["score"] = scores
feats["risk_level"] = F.risk_levels(scores, rules_all, cfg)
feats["hybrid"] = feats["risk_level"] != "low"
feats["rules_only"] = rules_all.any(axis=1)
budget = feats.loc[is_real, "hybrid"].mean()
feats["model_only"] = scores >= np.quantile(scores[is_real], 1 - budget)
limit = feats["category"].map(LIMITS)
feats["policy_rules"] = (feats["rules_only"] | (feats["amount_sar"] > limit) | (feats["is_weekend"] == 1)
                         | ((feats["is_round_amount"] == 1) & (feats["amount_sar"] >= 1000)))
print("Real requests flagged (%):", (feats.loc[is_real, SYSTEMS].mean() * 100).round(2).to_dict())

Real requests flagged (%): {'model_only': 4.47, 'rules_only': 3.09, 'policy_rules': 16.43, 'hybrid': 4.47}


In [17]:
# 8.17 One row per unseen case: flagged by each system? highest risk level, highest score, and the
# hybrid's reasons (8.3-8.4 method, only for flagged requests). Saved to a file FIRST, so the result
# is fixed before anyone knows what the patterns were.
import os
u = feats[~is_real].copy()
u["reason_codes"] = [[] for _ in range(len(u))]
flag_u = u["hybrid"]
if flag_u.any():                                       # reasons only if something was flagged
    Xu = F.model_input(u[flag_u], cfg)
    drop_u = pd.DataFrame(index=Xu.index)
    for feature, normal_value in NORMAL.items():
        what_if = Xu.copy()
        what_if[feature] = normal_value
        drop_u[feature] = u.loc[flag_u, "score"] - (-model.score_samples(what_if))
    u.loc[flag_u, "reason_codes"] = pd.Series(
        [reasons_for(u.loc[i], drop_u.loc[i], rules_all.loc[i]) for i in drop_u.index], index=drop_u.index)

RANK = {"low": 0, "medium": 1, "high": 2}
results = u.groupby("case_id").agg(
    requests=("expense_id", "size"),
    **{s: (s, "any") for s in SYSTEMS},
    risk_level=("risk_level", lambda s: max(s, key=RANK.get)),
    top_score=("score", "max"),
    reasons=("reason_codes", lambda s: sorted({c for lst in s for c in lst})),
).reset_index()
path = "../data/unseen/unseen_results.csv"          
if not os.path.exists(path):
    results.to_csv(path, index=False)
else:
    print("results already saved, not overwritten:", path)

print("Saved results for", len(results), "cases")
print("Cases flagged, BEFORE looking at the answers (%):")
print((results[SYSTEMS].mean() * 100).round(1))
print(results["risk_level"].value_counts().to_dict())
print("Highest unseen score:", round(results["top_score"].max(), 3),
      "| medium starts at:", round(cfg["cut_offs"]["medium_from"], 3))

results already saved, not overwritten: ../data/unseen/unseen_results.csv
Saved results for 151 cases
Cases flagged, BEFORE looking at the answers (%):
model_only       2.0
rules_only      25.2
policy_rules    35.1
hybrid          25.2
dtype: float64
{'low': 113, 'medium': 38}
Highest unseen score: 0.64 | medium starts at: 0.651


In [18]:
# 8.18 Only now: what were the patterns? Detection per pattern (suspicious cases only),
# false alarms on the normal controls, and each pattern's one-line description.
answers = pd.read_csv("../data/unseen/unseen_answers.csv")
res = results.merge(answers, on="case_id", how="left")
res["is_normal"] = res["is_normal"].astype(str).str.strip().str.lower().isin(["yes", "true", "1"])

suspicious = res[~res["is_normal"]]
table = suspicious.groupby("pattern").agg(cases=("case_id", "size"), **{s: (s, "mean") for s in SYSTEMS})
table[SYSTEMS] = (table[SYSTEMS] * 100).round(0)
table.loc["ALL suspicious"] = [len(suspicious)] + list((suspicious[SYSTEMS].mean() * 100).round(0))
print(table)
print("Normal controls flagged (%):", (res.loc[res["is_normal"], SYSTEMS].mean() * 100).round(0).to_dict())
print(answers.drop_duplicates("pattern")[["pattern", "description"]].to_string(index=False))

                      cases  model_only  rules_only  policy_rules  hybrid
pattern                                                                  
creeping_escalation      20         0.0         0.0           0.0     0.0
cross_day_duplicate      22         0.0         9.0           9.0     9.0
dormant_reactivation     22         0.0       100.0         100.0   100.0
fixed_monthly_repeat     20        15.0         0.0          75.0     0.0
micro_probe_claims       20         0.0         0.0           0.0     0.0
patterned_digits         22         0.0        64.0          64.0    64.0
ALL suspicious          126         2.0        30.0          42.0    30.0
Normal controls flagged (%): {'model_only': 0.0, 'rules_only': 0.0, 'policy_rules': 0.0, 'hybrid': 0.0}
             pattern                                                                                                                    description
      normal_control                                       An ordinary claim in th

### Round 1 → 3 new rules
Round 1 found 6 blind spots, all patterns that only show over time or in the digits. 3 are fixed with cheap rules
(receipt reused within 7 days, made-up-looking digits, first request after 90+ quiet days). 3 are documented limits:
creeping amounts (9% of real requests rise 3 times in a row by chance), tiny test claims (amounts are one-sided on
purpose, Step 7), fake monthly bills (look exactly like real subscriptions; needs the receipt).
These 6 are now "seen"; round 2 is a new unseen set.

In [19]:
# 8.19 The model itself doesn't change (same trees, same cut-offs), but the rules do, so the results change:
# new settings in model_config.json and a new version, so the backend knows to re-score.
# Note: notebook 06 (7.19) writes model_config.json without these settings; if you ever re-run 06, re-run this cell.
cfg["rules"].update({"recent_duplicate_days": 7, "long_break_days": 90})
cfg["model_version"] = "iforest-0.2"
with open("../models/model_config.json", "w", encoding="utf-8") as fh:
    json.dump(cfg, fh, indent=2)
print(cfg["model_version"], cfg["rules"])

iforest-0.2 {'split_min_part': 0.25, 'split_max_parts': 5, 'duplicate_min_amount': 1000, 'duplicate_max_copies': 3, 'many_other_from': 8, 'recent_duplicate_days': 7, 'long_break_days': 90}


In [20]:
# 8.20 How often each rule fires on the real history, and the risk levels before vs after.
# "before" = the same scores with only the 3 original rules.
NEW_RULES = ["duplicate_recent_request", "unusual_amount_digits", "returning_after_long_break"]
real_feats = F.build_features(requests)
real_scores = -model.score_samples(F.model_input(real_feats, cfg))
real_rules = F.apply_rules(real_feats, cfg)
print((real_rules.mean() * 100).round(2))

before = F.risk_levels(real_scores, real_rules.drop(columns=NEW_RULES), cfg)
after = F.risk_levels(real_scores, real_rules, cfg)
print("before:", (pd.Series(before).value_counts(normalize=True) * 100).round(2).to_dict())
print("after: ", (pd.Series(after).value_counts(normalize=True) * 100).round(2).to_dict())

possible_split_request        0.68
duplicate_request             1.23
many_requests_same_day        0.38
duplicate_recent_request      0.25
unusual_amount_digits         0.03
returning_after_long_break    0.65
dtype: float64
before: {'low': 96.38, 'medium': 2.62, 'high': 1.0}
after:  {'low': 95.52, 'medium': 3.47, 'high': 1.0}


In [21]:
# 8.21 The round-1 cases through the hybrid, before and after the 3 rules.
# These patterns are SEEN now (we built rules after looking), so this only shows what the fix covers;
# it is NOT an unseen score. Round 2 gives the honest number.
real = requests.assign(amount_original=requests["amount_usd"], case_id=0)
combined = pd.concat([real, unseen], ignore_index=True)
feats = F.build_features(combined, round_column="amount_original")
feats = feats.merge(combined[["expense_id", "case_id"]], on="expense_id", how="left")
is_real = feats["case_id"] == 0
scores = -model.score_samples(F.model_input(feats, cfg))
rules_all = F.apply_rules(feats, cfg)
feats["before"] = F.risk_levels(scores, rules_all.drop(columns=NEW_RULES), cfg) != "low"
feats["after"] = F.risk_levels(scores, rules_all, cfg) != "low"

by_case = feats[~is_real].groupby("case_id")[["before", "after"]].any().reset_index()
by_case = by_case.merge(answers, on="case_id")
print((by_case.groupby("pattern")[["before", "after"]].mean() * 100).round(0))

                      before  after
pattern                            
creeping_escalation      0.0    0.0
cross_day_duplicate      0.0    9.0
dormant_reactivation     0.0  100.0
fixed_monthly_repeat     0.0    0.0
micro_probe_claims       0.0    0.0
normal_control           0.0    0.0
patterned_digits         0.0   64.0


### Round 2: a new unseen set (after the 3 rules)
A second independent agent invented new patterns, with all 12 already-tested patterns on its "avoid" list.
Rule settings were fixed before scoring (receipt reuse: 7 days, ≥ 1,000 SAR). Results saved before opening the answers.

In [22]:
# 8.22 Same as 8.15, for the round-2 file. New ids (960001+) so nothing collides with round 1 or Part C.
unseen2 = pd.read_csv("../data/unseen_round2/unseen_requests.csv", parse_dates=["expense_date"])
unseen2["expense_id"] = np.arange(960001, 960001 + len(unseen2))
unseen2["amount_original"] = unseen2["amount_sar"]
print("Round 2:", len(unseen2), "requests in", unseen2["case_id"].nunique(), "cases")

Round 2: 299 requests in 167 cases


In [23]:
# 8.23 Same as 8.16, now with the 6 rules (config iforest-0.2).
real = requests.assign(amount_original=requests["amount_usd"], case_id=0)
combined = pd.concat([real, unseen2], ignore_index=True)
feats = F.build_features(combined, round_column="amount_original")
feats = feats.merge(combined[["expense_id", "case_id"]], on="expense_id", how="left")
is_real = feats["case_id"] == 0

scores = -model.score_samples(F.model_input(feats, cfg))
rules_all = F.apply_rules(feats, cfg)
feats["score"] = scores
feats["risk_level"] = F.risk_levels(scores, rules_all, cfg)
feats["hybrid"] = feats["risk_level"] != "low"
feats["rules_only"] = rules_all.any(axis=1)
budget = feats.loc[is_real, "hybrid"].mean()
feats["model_only"] = scores >= np.quantile(scores[is_real], 1 - budget)
limit = feats["category"].map(LIMITS)
feats["policy_rules"] = (feats["rules_only"] | (feats["amount_sar"] > limit) | (feats["is_weekend"] == 1)
                         | ((feats["is_round_amount"] == 1) & (feats["amount_sar"] >= 1000)))
print("Real requests flagged (%):", (feats.loc[is_real, SYSTEMS].mean() * 100).round(2).to_dict())

Real requests flagged (%): {'model_only': 4.48, 'rules_only': 3.09, 'policy_rules': 16.43, 'hybrid': 4.47}


In [24]:
# 8.24 Same as 8.17, saved to the round-2 folder.
u = feats[~is_real].copy()
u["reason_codes"] = [[] for _ in range(len(u))]
flag_u = u["hybrid"]
if flag_u.any():
    Xu = F.model_input(u[flag_u], cfg)
    drop_u = pd.DataFrame(index=Xu.index)
    for feature, normal_value in NORMAL.items():
        what_if = Xu.copy()
        what_if[feature] = normal_value
        drop_u[feature] = u.loc[flag_u, "score"] - (-model.score_samples(what_if))
    u.loc[flag_u, "reason_codes"] = pd.Series(
        [reasons_for(u.loc[i], drop_u.loc[i], rules_all.loc[i]) for i in drop_u.index], index=drop_u.index)

results2 = u.groupby("case_id").agg(
    requests=("expense_id", "size"),
    **{s: (s, "any") for s in SYSTEMS},
    risk_level=("risk_level", lambda s: max(s, key=RANK.get)),
    top_score=("score", "max"),
    reasons=("reason_codes", lambda s: sorted({c for lst in s for c in lst})),
).reset_index()
path = "../data/unseen_round2/unseen_results.csv"         
if not os.path.exists(path):
    results2.to_csv(path, index=False)
else:
    print("results already saved, not overwritten:", path)
    
print("Saved results for", len(results2), "cases")
print((results2[SYSTEMS].mean() * 100).round(1))
print(results2["risk_level"].value_counts().to_dict())
print(results2["reasons"].explode().value_counts().to_dict())

results already saved, not overwritten: ../data/unseen_round2/unseen_results.csv
Saved results for 167 cases
model_only      0.0
rules_only      0.6
policy_rules    1.8
hybrid          0.6
dtype: float64
{'low': 166, 'medium': 1}
{'returning_after_long_break': 1}


In [25]:
# 8.25 Same as 8.18, for round 2.
answers2 = pd.read_csv("../data/unseen_round2/unseen_answers.csv")
res2 = results2.merge(answers2, on="case_id", how="left")
res2["is_normal"] = res2["is_normal"].astype(str).str.strip().str.lower().isin(["yes", "true", "1"])

suspicious2 = res2[~res2["is_normal"]]
table2 = suspicious2.groupby("pattern").agg(cases=("case_id", "size"), **{s: (s, "mean") for s in SYSTEMS})
table2[SYSTEMS] = (table2[SYSTEMS] * 100).round(0)
table2.loc["ALL suspicious"] = [len(suspicious2)] + list((suspicious2[SYSTEMS].mean() * 100).round(0))
print(table2)
print("Normal controls flagged (%):", (res2.loc[res2["is_normal"], SYSTEMS].mean() * 100).round(0).to_dict())
print(answers2.drop_duplicates("pattern")[["pattern", "description"]].to_string(index=False))

                          cases  model_only  rules_only  policy_rules  hybrid
pattern                                                                      
colleague_receipt_reuse      20         0.0         0.0           0.0     0.0
daily_personal_meals         20         0.0         0.0           0.0     0.0
eid_holiday_purchase         20         0.0         0.0           0.0     0.0
quarter_end_budget_burn      20         0.0         0.0          10.0     0.0
retail_price_gadget          20         0.0         5.0           5.0     5.0
shared_bill_double_claim     22         0.0         0.0           0.0     0.0
weekend_stretched_trip       20         0.0         0.0           0.0     0.0
ALL suspicious              142         0.0         1.0           2.0     1.0
Normal controls flagged (%): {'model_only': 0.0, 'rules_only': 0.0, 'policy_rules': 0.0, 'hybrid': 0.0}
                 pattern                                                                                            

## Part E: Real-data checks
With the final settings (iforest-0.2): flag rate per month, per department, reason mix, and a 20-request sample
for a human check. Real flags are "unverified": PCard rows are unlabeled, so no precision or false-alarm rate is claimed (MF3).

In [26]:
# 8.26 Score the real history with the final config (6 rules) and give every flagged request its reasons
# (same 8.3-8.4 method).
real = F.build_features(requests)
real["score"] = -model.score_samples(F.model_input(real, cfg))
real_rules = F.apply_rules(real, cfg)
real["risk_level"] = F.risk_levels(real["score"], real_rules, cfg)
flagged = real["risk_level"] != "low"

X_flag = F.model_input(real[flagged], cfg)
drop = pd.DataFrame(index=X_flag.index)
for feature, normal_value in NORMAL.items():
    what_if = X_flag.copy()
    what_if[feature] = normal_value
    drop[feature] = real.loc[flagged, "score"] - (-model.score_samples(what_if))
real["reason_codes"] = [[] for _ in range(len(real))]
real.loc[flagged, "reason_codes"] = pd.Series(
    [reasons_for(real.loc[i], drop.loc[i], real_rules.loc[i]) for i in drop.index], index=drop.index)

print(cfg["model_version"], "|", real["risk_level"].value_counts().to_dict())
print("Flagged without a reason:", (real.loc[flagged, "reason_codes"].str.len() == 0).sum())
print((real.loc[flagged, "reason_codes"].explode().value_counts(normalize=True) * 100).round(1))

iforest-0.2 | {'low': 199658, 'medium': 7262, 'high': 2092}
Flagged without a reason: 0
reason_codes
large_amount                         19.4
duplicate_request                    16.0
weekend_expense                      12.8
possible_split_request                8.8
returning_after_long_break            8.5
amount_high_for_employee_category     7.8
unusual_category_for_employee         7.8
round_amount                          6.2
many_requests_same_day                5.0
duplicate_recent_request              3.3
amount_high_for_employee              3.0
amount_high_for_department            0.9
unusual_amount_digits                 0.4
Name: proportion, dtype: float64


In [27]:
# 8.27 High and total flagged (%) per month of the real year.
year = real[(real["expense_date"] >= "2025-06-29") & (real["expense_date"] <= "2026-06-27")].copy()
year["month"] = year["expense_date"].dt.to_period("M")
monthly = year.groupby("month").agg(
    requests=("expense_id", "size"),
    high=("risk_level", lambda s: (s == "high").mean() * 100),
    flagged=("risk_level", lambda s: (s != "low").mean() * 100),
)
print(monthly.round(2))

         requests  high  flagged
month                           
2025-06      1416  0.35     1.91
2025-07     17714  0.98     3.53
2025-08     17774  1.27     3.74
2025-09     19131  1.10     3.63
2025-10     18364  0.96     4.19
2025-11     15226  0.95     4.32
2025-12     12511  0.86     3.98
2026-01     17957  0.96     4.61
2026-02     17047  0.94     5.13
2026-03     18999  0.88     5.26
2026-04     18898  0.99     5.11
2026-05     16608  1.01     5.17
2026-06     16032  1.04     5.04


In [28]:
# 8.28 Flagged and high (%) per department: the 3 highest and 3 lowest, and the top reasons in the 2 highest.
by_dept = real.groupby("department_id").agg(
    requests=("expense_id", "size"),
    flagged=("risk_level", lambda s: (s != "low").mean() * 100),
    high=("risk_level", lambda s: (s == "high").mean() * 100),
).sort_values("flagged", ascending=False).round(2)
print(by_dept.iloc[[0, 1, 2, -3, -2, -1]])
for dept in by_dept.index[:2]:
    top = real.loc[(real["department_id"] == dept) & flagged, "reason_codes"].explode().value_counts(normalize=True)
    print(dept, (top * 100).round(0).head(3).to_dict())

               requests  flagged  high
department_id                         
20                 2112    10.89  1.56
17                 1462    10.81  1.98
7                  5862     6.65  1.86
3                 27573     3.63  0.89
8                  4376     3.29  0.80
6                  8951     2.37  0.84
20 {'duplicate_request': 30.0, 'weekend_expense': 19.0, 'many_requests_same_day': 11.0}
17 {'duplicate_request': 46.0, 'weekend_expense': 12.0, 'large_amount': 11.0}


In [29]:
# 8.29 10 high + 10 medium real requests, saved with an empty "verdict" column for a human check:
# write "suspicious", "looks normal" or "can't tell" for each. These are UNVERIFIED (no labels).
import os
os.makedirs("../data/evaluation", exist_ok=True)
sample = pd.concat([real[real["risk_level"] == "high"].sample(10, random_state=7),
                    real[real["risk_level"] == "medium"].sample(10, random_state=7)])
sample = sample[["expense_id", "requester_id", "department_id", "category", "amount_sar", "expense_date",
                 "risk_level", "reason_codes"]].assign(verdict="")
sample.to_csv("../data/evaluation/hand_check_sample.csv", index=False)
print(sample.drop(columns="verdict").head(10).to_string(index=False))

 expense_id  requester_id  department_id                category  amount_sar expense_date risk_level                                                       reason_codes
     120757          3817              6            IT & Telecom     3750.00   2025-07-28       high                                                     [round_amount]
     249328          5476             11          Transportation     1875.00   2026-02-08       high                                       [round_amount, large_amount]
     210393          5281              3            IT & Telecom    22471.20   2025-12-05       high                                    [large_amount, weekend_expense]
     174589          3900             10                  Travel     1720.28   2025-10-11       high                        [weekend_expense, amount_high_for_employee]
     299629          1053              1                  Travel     4694.81   2026-04-17       high                   [weekend_expense, unusual_category_for_em

In [30]:
# 8.29b The original shop and merchant type for the sample, for the human check only.
# orig_vendor / orig_mcc are NOT model inputs (Masroof has no vendor field); they only help a human judge.
lookup = requests[["expense_id", "orig_vendor", "orig_mcc"]]
print(sample.merge(lookup, on="expense_id")[["expense_id", "category", "amount_sar", "risk_level",
                                              "orig_vendor", "orig_mcc"]].to_string(index=False))

 expense_id                category  amount_sar risk_level               orig_vendor                                                orig_mcc
     120757            IT & Telecom     3750.00       high   MOTOROLA, INC. - ONLINE         TELECOMMUNICATION EQUIPMENT AND TELEPHONE SALES
     249328          Transportation     1875.00       high   ACE ALTERNATOR & STARTE                           AUTOMOTIVE TOP AND BODY SHOPS
     210393            IT & Telecom    22471.20       high      DMI DELL HIGHER EDUC      COMPUTERS, COMPUTER PERIPHERAL EQUIPMENT, SOFTWARE
     174589                  Travel     1720.28       high    OMNI DALLAS CON CTR HT                                      OMNI INTERNATIONAL
     299629                  Travel     4694.81       high  RENAISSANCE HOTELS 9671V                                      RENAISSANCE HOTELS
     179358 Equipment & Maintenance   125473.05       high                 REXEL2441                          ELECTRICAL PARTS AND EQUIPMENT
     289143  

### 8.30 Final cut-offs: trim the model's medium share to 0.5%
With the 3 new rules, about 5% of recent requests were flagged, which is more than the ~4% proposed to the team.
The model's "medium" band is cut from the next 1% to the next 0.5%. High stays the model's top 1%,
and any rule still gives medium. On the seen planted cases the hybrid still catches 86%.

In [31]:
# 8.30 Trim the model's medium share 1% → 0.5% (high stays the top 1%). medium_from = the score above which
# the model's top 1.5% of the TRAINING data lies; high_from is unchanged (check it matches).
train = pd.read_csv("../data/processed/features.csv", parse_dates=["expense_date"])
train_scores = -model.score_samples(F.model_input(train, cfg))
print("high_from check:", round(float(np.quantile(train_scores, 0.99)), 4), "=", round(cfg["cut_offs"]["high_from"], 4))
cfg["cut_offs"]["model_flag_share"] = 0.015
cfg["cut_offs"]["medium_from"] = float(np.quantile(train_scores, 1 - cfg["cut_offs"]["model_flag_share"]))
with open("../models/model_config.json", "w", encoding="utf-8") as fh:
    json.dump(cfg, fh, indent=2)
real["risk_level"] = F.risk_levels(real["score"], real_rules, cfg)
print(cfg["model_version"], "| medium from", round(cfg["cut_offs"]["medium_from"], 4), "| high from", round(cfg["cut_offs"]["high_from"], 4))
print((real["risk_level"].value_counts(normalize=True) * 100).round(2).to_dict())

high_from check: 0.6639 = 0.6639
iforest-0.2 | medium from 0.6513 | high from 0.6639
{'low': 95.52, 'medium': 3.47, 'high': 1.0}


### 8.31 Evaluation summary
All four systems are rerun with the final settings: the model alone, the rules alone, policy rules without ML, and the hybrid.
Every Step 8 number goes into `models/evaluation_summary.json` (real-data flag rates, seen planted, both unseen rounds,
the blind hand-check, and the limits). Real requests are unlabeled, so flags are "unverified" (MF3).

In [32]:
# 8.31 Final comparison of the 4 systems + save all Step 8 results in one file
def four_systems(extra):
    # build features for real history + the planted rows together (planted rows get real history before them)
    real_part = requests.assign(amount_original=requests["amount_usd"], case_id=0)
    combined = pd.concat([real_part, extra], ignore_index=True)
    f = F.build_features(combined, round_column="amount_original")
    f = f.merge(combined[["expense_id", "case_id"]], on="expense_id", how="left")
    is_real = f["case_id"] == 0
    s = -model.score_samples(F.model_input(f, cfg))
    r = F.apply_rules(f, cfg)
    f["hybrid"] = F.risk_levels(s, r, cfg) != "low"
    f["rules_only"] = r.any(axis=1)
    # model alone gets the SAME flag budget as the hybrid on real data (fair comparison, MF4)
    f["model_only"] = s >= np.quantile(s[is_real], 1 - f.loc[is_real, "hybrid"].mean())
    # simple policy rules without ML: our rules + over the limit + weekend + big round amount
    f["policy_rules"] = (f["rules_only"] | (f["amount_sar"] > f["category"].map(LIMITS)) | (f["is_weekend"] == 1)
                         | ((f["is_round_amount"] == 1) & (f["amount_sar"] >= 1000)))
    flag_rate = (f.loc[is_real, SYSTEMS].mean() * 100).round(2).to_dict()   # % of real requests flagged
    cases = f[~is_real].groupby("case_id")[SYSTEMS].any()                    # a case is caught if any of its rows is flagged
    return flag_rate, cases

pct = lambda x: round(float(x) * 100, 1)
seen_rate, seen_cases = four_systems(planted)
seen_cases = seen_cases.join(planted.groupby("case_id")[["pattern", "spreads"]].first())
round2_rate, round2_cases = four_systems(unseen2)
round2_cases = round2_cases.join(pd.read_csv("../data/unseen_round2/unseen_answers.csv").set_index("case_id"))
round2_cases["is_normal"] = round2_cases["is_normal"].astype(str).str.strip().str.lower().isin(["yes", "true", "1"])
hand = pd.read_csv("../data/evaluation/hand_check_review.csv")

# review load: high + medium per department per week, Jan–Jun 2026 (most recent months)
year = real[(real["expense_date"] >= "2026-01-01") & (real["expense_date"] <= "2026-06-27")].copy()
year["week"] = year["expense_date"].dt.to_period("W-SAT")
weekly = (year.groupby(["department_id", "week"])
              .agg(high=("risk_level", lambda s: (s == "high").sum()), medium=("risk_level", lambda s: (s == "medium").sum()))
              .groupby("department_id").mean())
sizes = real.groupby("department_id").size()

summary = {
    "model_version": cfg["model_version"],
    "design": "hybrid: Isolation Forest (5 one-sided features) + 6 rules; high = model top 1%, medium = model next 0.5% or any rule",
    "real_data": {
        "requests": int(len(real)),
        "risk_levels_pct": {k: pct(v) for k, v in real["risk_level"].value_counts(normalize=True).items()},
        "flagged_pct_jan_jun_2026": pct((year["risk_level"] != "low").mean()),
        "per_week_biggest_department": weekly.loc[sizes.idxmax()].round(1).to_dict(),
        "per_week_median_department": weekly.median().round(1).to_dict(),
        "note": "unlabeled data: flags are unverified, no precision or false-alarm rate claimed (MF3)",
    },
    "seen_planted": {
        "cases": int(len(seen_cases)),
        "flagged_pct_on_real": seen_rate,
        "caught_pct_by_pattern": (seen_cases.groupby("pattern")[SYSTEMS].mean() * 100).round(0).to_dict("index"),
        "caught_pct_all": (seen_cases[SYSTEMS].mean() * 100).round(0).to_dict(),
        "too_big_by_spreads": (seen_cases.dropna(subset=["spreads"]).groupby("spreads")[SYSTEMS].mean() * 100).round(0).to_dict("index"),
    },
    "unseen_round1": {
        "made_by": "independent AI agent with no access to the model design",
        "result_before_fix": "hybrid 0/126 suspicious cases, model alone 2%, policy rules 12%; 0/25 normal controls flagged",
        "patterns": ["creeping_escalation", "cross_day_duplicate", "dormant_reactivation", "fixed_monthly_repeat",
                     "micro_probe_claims", "patterned_digits"],
        "fix": "3 rules added: duplicate_recent_request, unusual_amount_digits, returning_after_long_break (now seen)",
    },
    "unseen_round2": {
        "made_by": "a second independent AI agent; all 12 earlier patterns excluded",
        "flagged_pct_on_real": round2_rate,
        "suspicious_cases": int((~round2_cases["is_normal"]).sum()),
        "caught_pct_suspicious": (round2_cases.loc[~round2_cases["is_normal"], SYSTEMS].mean() * 100).round(1).to_dict(),
        "normal_controls_flagged_pct": (round2_cases.loc[round2_cases["is_normal"], SYSTEMS].mean() * 100).round(1).to_dict(),
        "caught_pct_by_pattern": (round2_cases[~round2_cases["is_normal"]].groupby("pattern")[SYSTEMS].mean() * 100).round(0).to_dict("index"),
    },
    "blind_hand_check": {
        "reviewer": "independent AI agent, did not know which requests were flagged",
        "verdicts_by_ai_level": pd.crosstab(hand["risk_level"].fillna("low"), hand["verdict"]).to_dict("index"),
    },
    "limits": ["judges one request at a time (+ that day and that person's history)",
               "can't see trends or recurring patterns over weeks/months",
               "can't compare employees with each other (shared / reused receipts)",
               "no holiday calendar; no receipt checking (MF5)",
               "trained on US purchase-card data: several purchases a day are normal there, rules fire more than they would in Masroof"],
}
with open("../models/evaluation_summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)   # ensure_ascii=False keeps Arabic readable
print(json.dumps({k: summary[k] for k in ("real_data",)}, indent=2))
print(summary["seen_planted"]["caught_pct_all"], summary["unseen_round2"]["caught_pct_suspicious"])

{
  "real_data": {
    "requests": 209012,
    "risk_levels_pct": {
      "low": 95.5,
      "medium": 3.5,
      "high": 1.0
    },
    "flagged_pct_jan_jun_2026": 5.1,
    "per_week_biggest_department": {
      "high": 11.3,
      "medium": 51.4
    },
    "per_week_median_department": {
      "high": 0.7,
      "medium": 3.8
    },
    "note": "unlabeled data: flags are unverified, no precision or false-alarm rate claimed (MF3)"
  }
}
{'model_only': 55.0, 'rules_only': 42.0, 'policy_rules': 94.0, 'hybrid': 86.0} {'model_only': 0.0, 'rules_only': 0.7, 'policy_rules': 2.1, 'hybrid': 0.7}
